In [1]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import scipy
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import interp1d
import lmfit
import ray
import os
import timeit

from bio_optics.water import absorption, attenuation, backscattering, scattering, lee
from bio_optics.atmosphere import downwelling_irradiance
from bio_optics.models import hereon, model
from bio_optics.helper import resampling, utils, owt, indices, plotting
from bio_optics.water import fluorescence

# Forward Bi-Model  
- Use insitu data and replace values from inversion (the results ...allOWTS... are organised in the original order)
- model Spectra with Bi bio-optical model

## Setup Parameter Functions and global variables

In [2]:
def set_default_parameters(AlgaeGroupType='Standardv3'):
    ## by Default:
    # - all Phytoplankton groups are value=0, min=0, max=1000 (C_5=10, C_7=1), vary = False
    # - C_Y value=0, min=0, max=30, vary=True
    # - C_ism value=0, min=0, max=100, vary=True
    # - fluorescence all vary = False
    # - offset vary = False
    params = lmfit.Parameters()                             # v3 ,      SummerBloom
    params.add('C_0', value=0, min=0, max=100, vary=False)  # Diatoms
    params.add('C_1', value=0, min=0, max=100, vary=False)  # green
    params.add('C_2', value=0, min=0, max=100, vary=False)  # cryptophyte
    params.add('C_3', value=0, min=0, max=100, vary=False)  # cyano blue
    params.add('C_4', value=0, min=0, max=100, vary=False)  # cyano red
    params.add('C_5', value=0, min=0, max=10, vary=False)   # coccolithophores , Phaeocystis: other ranges
    params.add('C_6', value=0, min=0, max=100, vary=False)  # dinoflagellates
    params.add('C_7', value=0, min=0, max=1, vary=False)    # case-1  , Noctiluca: other ranges
    params.add('C_Y', value=0, min=0, max=30, vary=True)
    params.add('C_ism', value=0, min=0, max=100, vary=True)
    params.add('L_fl_lambda0', value=0, min=0, max=0.2, vary=False)
    params.add('L_fl_phycocyanin', value=0, min=0, max=0.2, vary=False)
    params.add('L_fl_phycoerythrin', value=0, min=0, max=0.2, vary=False)
    params.add('b_ratio_C_0', value=0.0058, vary=False)  # Diatoms
    params.add('b_ratio_C_1', value=0.007, vary=False)  # green
    params.add('b_ratio_C_2', value=0.0042, vary=False)  # cryptophyte
    params.add('b_ratio_C_3', value=0.0082, vary=False)  # cyano blue
    params.add('b_ratio_C_4', value=0.001, vary=False)  # cyano red
    # if AlgaeGroupType == 'NSSummerBloomsv3':
    if AlgaeGroupType == 'Summer':
        params.add('b_ratio_C_5', value=0.0034, vary=False)  # Phaeocystis: change to 0.0034
    else:
        params.add('b_ratio_C_5', value=0.0129, vary=False)  # coccolithophores , Phaeocystis: change to 0.0034
    params.add('b_ratio_C_6', value=0.0209, vary=False)  # dinoflagellates
    # if AlgaeGroupType == 'NSSummerBloomsv3':
    if AlgaeGroupType == 'Summer':
        params.add('b_ratio_C_7', value=0.0209, vary=False)  # Noctiluca: change to 0.0209
    else:
        params.add('b_ratio_C_7', value=0.0109, vary=False)  # case-1 , Noctiluca: change to 0.0209
    params.add('b_ratio_md', value=0.0216, min=0.021, max=0.3756, vary=True)  # max=0.0756
    params.add('b_ratio_bd', value=0.0216, min=0.021, max=0.3756, vary=True)  # max=0.0756
    # params.add('b_ratio_d', value=0.0216, min=0.021, max=0.3756, vary=True)
    params.add('A_md', value=13.4685e-3, vary=False)
    params.add('A_bd', value=0.3893e-3, vary=False)
    params.add('S_md', value=10.3845e-3, vary=False)
    params.add('S_bd', value=15.7621e-3, vary=False)
    params.add('S_cdom', value=0.0185, min=0.005, max=0.032, vary=True) # test Baltic: min =0.01
    params.add('C_md', value=12.1700e-3, vary=False)
    params.add('C_bd', value=0.9994e-3, vary=False)
    params.add('K', value=0, min=0, vary=False)
    params.add('lambda_0_cdom', value=440, vary=False)
    params.add('lambda_0_md', value=550, vary=False)
    params.add('lambda_0_bd', value=550, vary=False)
    params.add('lambda_0_c_d', value=550, vary=False)
    params.add('lambda_0_phy', value=676, vary=False)
    params.add('gamma_d', value=0.3835, vary=False)
    params.add('x0', value=1, vary=False)
    params.add('x1', value=10, vary=False)
    params.add('x2', value=-1.3390, min=-1.3390 - 0.0618, max=-1.3390 + 0.0618, vary=False)
    params.add('A', value=0.0237, vary=False)
    params.add('E0', value=1, vary=False)
    params.add('E1', value=0.8987, vary=False)
    params.add('W', value=0.75, vary=False)
    params.add('fwhm1', value=25, vary=False)
    params.add('fwhm2', value=50, vary=False)
    params.add('fwhm_phycocyanin', value=20, vary=False)
    params.add('fwhm_phycoerythrin', value=20, vary=False)
    params.add('lambda_C1', value=685, vary=False)
    params.add('lambda_C2', value=730, vary=False)
    params.add('lambda_C_phycocyanin', value=644, vary=False)
    params.add('lambda_C_phycoerythrin', value=573, vary=False)
    params.add('double', value=True, vary=False)
    params.add('interpolate', value=True, vary=False)
    params.add("Gw0", value=0.05881474, vary=False)
    params.add("Gw1", value=0.05062697, vary=False)
    params.add("Gp0", value=0.03997009, vary=False)
    params.add("Gp1", value=0.1398902, vary=False)
    params.add('error_method', value=0, vary=False)
    params.add('theta_sun', value=np.radians(30), min=np.radians(0), max=np.radians(90), vary=False)
    params.add('theta_view', value=np.radians(1e-10), min=np.radians(1e-10), max=np.radians(90), vary=False)
    params.add('n1', value=1, vary=False)
    params.add('n2', value=1.33, vary=False)
    params.add('kappa_0', value=1.0546, vary=False)
    params.add('fresh', value=False, vary=False)
    params.add('T_W', value=25, min=0, max=40, vary=False)
    params.add('T_W_0', value=20, vary=False)
    params.add('P', value=1013.25, vary=False)
    params.add('AM', value=1, vary=False)
    params.add('RH', value=60, vary=False)
    params.add('H_oz', value=0.38, vary=False)
    params.add('WV', value=2.5, vary=False)
    params.add('alpha', value=1.317, vary=False)
    params.add('beta', value=0.2606, vary=False)
    params.add('g_dd', value=0.02, min=-1, max=10, vary=False)  # glint correction
    params.add('g_dsr', value=1 / np.pi, min=0, max=10, vary=False)  # glint correction
    params.add('g_dsa', value=1 / np.pi, min=0, max=10, vary=False)  # glint correction
    params.add('d_r', value=0, min=0, max=0.1, vary=False)
    params.add('f_dd', value=1, vary=False)
    params.add('f_ds', value=1, vary=False)
    params.add('offset', value=0, min=-0.1, max=0.1, vary=False)
    params.add('fit_surface', value=False, vary=False)
    return params


def set_parameters_byDict(pDict, params):
    for key in pDict.keys():
        thisD = pDict[key]
        params.add(key, value=thisD['value'], min=thisD['min'], max=thisD['max'], vary=True)
    return params

In [3]:
# Define wavelength range and sampling rate
wavelengths=np.arange(400,900, 5)

# Select iop-model setup!
AlgaeGroupType = 'HEREON' #'Standard' # 'Standardv2': HEREON, 'Standardv3': 'Standard', NSSummerBloomsv3': 'Summer'


# global inputs that don't change with fit params
a_md_spec_res = absorption.a_md_spec(wavelengths=wavelengths)
a_bd_spec_res = absorption.a_bd_spec(wavelengths=wavelengths)
a_w_res = resampling.resample_a_w(wavelengths=wavelengths)
if AlgaeGroupType == 'HEREON':
    a_i_spec_res = resampling.resample_a_i_spec_EnSAD(wavelengths=wavelengths) # prototype v2, PACEv2
    b_i_spec_res = resampling.resample_b_i_spec_EnSAD(wavelengths=wavelengths) # prototype v2, PACEv2
if AlgaeGroupType == 'Standard':
    a_i_spec_res = resampling.resample_a_i_spec_EnSAD_Standardv3(wavelengths=wavelengths) # prototype v3, PACEv3
    b_i_spec_res = resampling.resample_b_i_spec_EnSAD_Standardv3(wavelengths=wavelengths) # prototype v3, PACEv3
if AlgaeGroupType == 'Summer':
    a_i_spec_res = resampling.resample_a_i_spec_EnSAD_SummerBloomsv3(wavelengths=wavelengths) # prototype v3, PACEv3
    b_i_spec_res = resampling.resample_b_i_spec_EnSAD_SummerBloomsv3(wavelengths=wavelengths) # prototype v3, PACEv3
b_bw_res = backscattering.b_bw(wavelengths=wavelengths, fresh=False)

da_W_div_dT_res = resampling.resample_da_W_div_dT(wavelengths=wavelengths)
h_C_res = fluorescence.h_C_double(wavelengths=wavelengths, W=0.75)
h_C_phycocyanin_res = fluorescence.h_C(wavelengths=wavelengths, fwhm=20, lambda_C=644)
h_C_phycoerythrin_res =fluorescence.h_C(wavelengths=wavelengths, fwhm=20, lambda_C=573)
omega_d_lambda_0_res = attenuation.omega_d_lambda_0()

E_0_res = resampling.resample_E_0(wavelengths=wavelengths)
a_oz_res = resampling.resample_a_oz(wavelengths=wavelengths)
a_ox_res = resampling.resample_a_ox(wavelengths=wavelengths)
a_wv_res = resampling.resample_a_wv(wavelengths=wavelengths)
n2_res = resampling.resample_n(wavelengths=wavelengths)

E_dd_res = downwelling_irradiance.E_dd(wavelengths=wavelengths)
E_dsa_res = downwelling_irradiance.E_dsa(wavelengths=wavelengths)
E_dsr_res = downwelling_irradiance.E_dsr(wavelengths=wavelengths)
E_d_res = E_dd_res + E_dsa_res + E_dsr_res

## Read full inversion IOP file

In [5]:
outpath = "Z:\projects\\ongoing\HEATWISE\\sharepoint\WP2_workspace\Water Quality\FullInversion\\bio-optics_forward\\"
# path = "Z:\projects\\ongoing\HEATWISE\\sharepoint\WP2_workspace\Water Quality\Helsinki\\input_simulations\\"
path = "Z:\projects\\ongoing\HEATWISE\\sharepoint\WP2_workspace\Water Quality\FullInversion\\"
location = 'Helsinki'

fnames = os.listdir(path)
iopFnames = [fn for fn in fnames if fn.startswith('inverted_IOP') and fn.endswith('allOWTs.txt') and location in fn]
print(iopFnames)

# paramDF = pd.read_csv(path + iopFnames[0], header=0, sep='\t')
paramDF = pd.read_csv(path + iopFnames[0], header=0)
if 'date' in paramDF.columns.values:
    paramDF = paramDF.drop('date', axis=1)
# print(paramDF.iloc[0,:])
# print(paramDF.columns.values)
paramList = ['C_0', 'C_1', 'C_2', 'C_3', 'C_4', 'C_5', 'C_6', 'C_7', 'C_Y', 'C_ism', 'L_fl_lambda0', 'L_fl_phycocyanin',
             'L_fl_phycoerythrin', 'b_ratio_C_0', 'b_ratio_C_1', 'b_ratio_C_2', 'b_ratio_C_3', 'b_ratio_C_4',
             'b_ratio_C_5', 'b_ratio_C_6', 'b_ratio_C_7', 'b_ratio_md', 'b_ratio_bd', 'A_md', 'A_bd', 'S_md', 'S_bd',
             'S_cdom', 'C_md', 'C_bd', 'K', 'lambda_0_cdom', 'lambda_0_md', 'lambda_0_bd', 'lambda_0_c_d', 'lambda_0_phy',
             'gamma_d', 'x0', 'x1', 'x2', 'A', 'E0', 'E1', 'W', 'fwhm1', 'fwhm2', 'fwhm_phycocyanin', 'fwhm_phycoerythrin',
             'lambda_C1', 'lambda_C2', 'lambda_C_phycocyanin', 'lambda_C_phycoerythrin', 'double', 'interpolate',
             'Gw0', 'Gw1', 'Gp0', 'Gp1', 'error_method', 'theta_sun', 'theta_view', 'n1', 'n2', 'kappa_0', 'fresh',
             'T_W', 'T_W_0', 'offset', 'fit_surface']
# for v in paramDF.columns.values:
#     if len(np.unique(paramDF[v])) > 1 and v != 'C_phy':
#         paramList.append(v)
paramDF.drop(columns = ['C_phy'], inplace=True)

['inverted_IOP_bio_optics_HEREONfull_CHIME_sim_HEATWISE_Helsinki_SiljaSerenade_allOWTs.txt']


In [6]:
paramDF

,Unnamed: 0,C_0,C_1,C_2,C_3,C_4,C_5,C_6,C_7,C_Y,...,alpha,beta,g_dd,g_dsr,g_dsa,d_r,f_dd,f_ds,offset,fit_surface
0,0,8.009653e-02,0.0,0.0,0.019903,0.0,0.0,0.0,0.0,0.474351,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
1,1,7.889359e-02,0.0,0.0,0.021106,0.0,0.0,0.0,0.0,0.494600,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
2,2,9.681224e-02,0.0,0.0,0.003188,0.0,0.0,0.0,0.0,0.445048,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
3,3,8.901558e-02,0.0,0.0,0.010984,0.0,0.0,0.0,0.0,0.446575,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
4,4,8.913490e-02,0.0,0.0,0.010865,0.0,0.0,0.0,0.0,0.425791,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,287,1.944951e-01,0.0,0.0,0.705969,0.0,0.0,0.0,0.0,0.529969,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
288,288,1.940299e-36,0.0,0.0,1.000299,0.0,0.0,0.0,0.0,0.491312,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
289,289,7.785141e-02,0.0,0.0,0.022149,0.0,0.0,0.0,0.0,0.536810,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0
290,290,4.746554e-47,0.0,0.0,1.028825,0.0,0.0,0.0,0.0,0.534592,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,0.0,0.0


## Read insitu data and exchange values

In [9]:
## Exchange columns with input and order them like inversion results by OWT!
# OWTList = [ '1', '2', '3a_g', '3a_y', '3b', '4a_g', '4a_y', '4b', '5a', '5b', '6', '7']
regionName = 'Helsinki_SS'

metaDict = {
    'elbe_bunthaus' : {'path': "Z:\projects\ongoing\AQUATIME\sharepoint\WP2 Representative Dataset\\RTM_simulations\Simulated Spectra\elbe_bunthaus\\",
                    'datasetName' : 'CHIME_sim_AQUATIME_elbe_bunthaus'},
    'dalaro-2': {'path' : "Z:\projects\ongoing\AQUATIME\sharepoint\WP2 Representative Dataset\\RTM_simulations\Simulated Spectra\dalaro-2\\",
             'datasetName': 'CHIME_sim_AQUATIME_dalaro-2'},
    'pyhajarvi': {'path' : "Z:\projects\ongoing\AQUATIME\sharepoint\WP2 Representative Dataset\\RTM_simulations\Simulated Spectra\pyhajarvi\\",
                  'datasetName' :'CHIME_sim_AQUATIME_pyhajarvi'},
    'elbe_seemannshöft' :{ 'path': "Z:\projects\ongoing\AQUATIME\sharepoint\WP2 Representative Dataset\\RTM_simulations\Simulated Spectra\elbe_seemannshöft\\",
                           'datasetName': 'CHIME_sim_AQUATIME_elbe-seemannshöft' },
    'oder_frankfurt': {'path': "Z:\projects\ongoing\AQUATIME\sharepoint\WP2 Representative Dataset\\RTM_simulations\Simulated Spectra\oder_frankfurt\\",
                       'datasetName': 'CHIME_sim_AQUATIME_oder_frankfurt'},
    'oder_hohenwutzen': {'path': "Z:\projects\ongoing\AQUATIME\sharepoint\WP2 Representative Dataset\\RTM_simulations\Simulated Spectra\oder_hohenwutzen\\",
                         'datasetName': 'CHIME_sim_AQUATIME_oder_hohenwutzen'},
    'Helsinki_SS': {
        'path': "Z:\projects\ongoing\HEATWISE\sharepoint\WP2_workspace\Water Quality\\representative_dataset\helsinki_siljaserenade\\", 
        'datasetName': '',
        'iop_path': "Z:\projects\ongoing\HEATWISE\sharepoint\WP2_workspace\Water Quality\Helsinki\input_simulations\\",
        'datasetNameIOP': 'helsinki_silja_serenade'},
    'Helsinki_FM': {
        'path': "Z:\projects\ongoing\HEATWISE\sharepoint\WP2_workspace\Water Quality\\representative_dataset\helsinki_finnmaid\\"},
    'Mueggelsee': {'path' : "Z:\projects\ongoing\HEATWISE\sharepoint\WP2_workspace\Water Quality\\representative_dataset\\berlin_muggelsee\\"}
}

# path = metaDict[regionName]['path']

iop_path = metaDict[regionName]['iop_path']
datasetNameIOP = metaDict[regionName]['datasetNameIOP']

fname = os.listdir(iop_path)
fname = [fn for fn in fname if 'IOP' in fn and fn.startswith(datasetNameIOP) and 'v0.1b' in fn]
print(fname)
iop_insitu = pd.read_csv(iop_path+ fname[0], sep=',')

['helsinki_silja_serenade_IOP_v0.1b.csv']


In [10]:
## for replacement:
IOPDict = {
    'C_0': 'Ckie [µg/l]', 
    'C_1': None, 
    'C_2': None, #'Ccry [µg/l]'
    'C_3': 'Cbl [µg/l]',
    'C_4': None, 
    'C_5': None, 
    'C_6': None, 
    'C_Y': 'ag(440) [1/m]', 
    'C_ism': 'NAP [mg/l]', 
    'S_md': 'Snap [1/nm]', 
    'S_cdom' : 'Sg [1/nm]' 
}

for key in IOPDict.keys():
    if IOPDict[key] is None:
        paramDF[key] = 0.
    else:
        y = iop_insitu[IOPDict[key]].values
        print(key, y[:10])
        ID = np.isnan(y)
        if np.sum(ID)>0:
            y[ID] =  0.
        paramDF[key] = y

C_0 [ 2.18066568  2.2390477   2.30585599  2.46956639  2.26462745  2.5562366
  4.30248087  3.39752616  3.77443458 16.86163655]
C_3 [nan nan nan nan nan nan nan nan nan  0.]
C_Y [0.18806794 0.21960659 0.25479349 0.17620504 0.20148655 0.2197692
 0.23084623 0.26362199 0.22461713 0.34047606]
C_ism [5.15614581 5.16168461 4.55650864 4.36466726 4.63792096 5.38923295
 5.55998239 5.04223761 5.20335594 3.42606311]
S_md [0.011 0.011 0.011 0.011 0.011 0.011 0.011 0.011 0.011 0.011]
S_cdom [0.02147749 0.02132791 0.02116282 0.02153411 0.02141358 0.02132685
 0.02127453 0.02112161 0.02130399 0.02076658]


In [11]:
@ray.remote
def simulate_chunk(
                   paramDF_chunk,  # iops and variables from Inversion
                   wavelengths,
                   a_md_spec_res,
                   a_bd_spec_res,
                   a_w_res,
                   a_i_spec_res,
                   b_bw_res,
                   b_i_spec_res,
                   h_C_res,
                   h_C_phycocyanin_res,
                   h_C_phycoerythrin_res,
                   da_W_div_dT_res,
                   E_0_res,
                   a_oz_res,
                   a_ox_res,
                   a_wv_res,
                   E_dd_res,
                   E_dsa_res,
                   E_dsr_res,
                   E_d_res,
                   n2_res):

    params = set_default_parameters(AlgaeGroupType)

    for i in np.arange(paramDF_chunk.shape[0]):
        for p in paramList:
            # Change parameters object accordingly.
            params.add(p, value=paramDF_chunk[p].values[i])

        if i == 0:
            print(params)
            R_rs_sim = hereon.forward(parameters=params,
                                      wavelengths=wavelengths,
                                      a_md_spec_res=a_md_spec_res,
                                      a_bd_spec_res=a_bd_spec_res,
                                      a_w_res=a_w_res,
                                      a_i_spec_res=a_i_spec_res,
                                      b_bw_res=b_bw_res,
                                      b_i_spec_res=b_i_spec_res,
                                      h_C_res=h_C_res,
                                      h_C_phycocyanin_res=h_C_phycocyanin_res,
                                      h_C_phycoerythrin_res=h_C_phycoerythrin_res,
                                      da_W_div_dT_res=da_W_div_dT_res,
                                      E_0_res=E_0_res,
                                      a_oz_res=a_oz_res,
                                      a_ox_res=a_ox_res,
                                      a_wv_res=a_wv_res,
                                      E_dd_res=E_dd_res,
                                      E_dsa_res=E_dsa_res,
                                      E_dsr_res=E_dsr_res,
                                      E_d_res=E_d_res,
                                      n2_res=n2_res,
                                      Ls_Ed=[])
        else:
            R_rs_sim = np.vstack((R_rs_sim, hereon.forward(parameters=params,
                                                           wavelengths=wavelengths,
                                                           a_md_spec_res=a_md_spec_res,
                                                           a_bd_spec_res=a_bd_spec_res,
                                                           a_w_res=a_w_res,
                                                           a_i_spec_res=a_i_spec_res,
                                                           b_bw_res=b_bw_res,
                                                           b_i_spec_res=b_i_spec_res,
                                                           h_C_res=h_C_res,
                                                           h_C_phycocyanin_res=h_C_phycocyanin_res,
                                                           h_C_phycoerythrin_res=h_C_phycoerythrin_res,
                                                           da_W_div_dT_res=da_W_div_dT_res,
                                                           E_0_res=E_0_res,
                                                           a_oz_res=a_oz_res,
                                                           a_ox_res=a_ox_res,
                                                           a_wv_res=a_wv_res,
                                                           E_dd_res=E_dd_res,
                                                           E_dsa_res=E_dsa_res,
                                                           E_dsr_res=E_dsr_res,
                                                           E_d_res=E_d_res,
                                                           n2_res=n2_res,
                                                           Ls_Ed=[])
                                  ))

    # print(R_rs_sim.shape)
    return R_rs_sim


In [14]:
if paramDF.shape[0] < 300:
    num_chunks = 1
elif paramDF.shape[0] < 1000:
    num_chunks = 3
else:
    num_chunks = 9  # Number of chunks to split the DF into

chunk_size = paramDF.shape[0] // num_chunks  # Size of each chunk
chunks = [paramDF.iloc[i:i + chunk_size, :] for i in range(0, paramDF.shape[0], chunk_size)]  # Split the DF into chunks
while chunks[-1].shape[0] == 1:
    num_chunks -= 1
    chunk_size = paramDF.shape[0] // num_chunks  # Size of each chunk
    chunks = [paramDF.iloc[i:i + chunk_size, :] for i in range(0, paramDF.shape[0], chunk_size)]  # Split the DF into chunks


print(chunks[-1].shape[0])
print('chunks N', len(chunks))
ray.shutdown()

start = timeit.default_timer()

# Parallelize the processing of the chunks using ray
chunk_refs = [ray.put(chunk) for chunk in chunks]  # Put the chunks into the object store
result_refs = [simulate_chunk.remote(chunk_ref,
                                   # params=params,
                                   wavelengths=wavelengths,
                                   a_md_spec_res=a_md_spec_res,
                                   a_bd_spec_res=a_bd_spec_res,
                                   a_w_res=a_w_res,
                                   a_i_spec_res=a_i_spec_res,
                                   b_bw_res=b_bw_res,
                                   b_i_spec_res=b_i_spec_res,
                                   h_C_res=h_C_res,
                                   h_C_phycocyanin_res=h_C_phycocyanin_res,
                                   h_C_phycoerythrin_res=h_C_phycoerythrin_res,
                                   da_W_div_dT_res=da_W_div_dT_res,
                                   E_0_res=E_0_res,
                                   a_oz_res=a_oz_res,
                                   a_ox_res=a_ox_res,
                                   a_wv_res=a_wv_res,
                                   E_dd_res=E_dd_res,
                                   E_dsa_res=E_dsa_res,
                                   E_dsr_res=E_dsr_res,
                                   E_d_res=E_d_res,
                                   n2_res=n2_res) for chunk_ref in chunk_refs]  # Process the chunks in parallel

results = ray.get(result_refs)

# Concatenate the results from the processed chunks
processed_data = np.concatenate(results)

stop = timeit.default_timer()
print('Time: ', stop - start)


R_rs_sim = pd.DataFrame(processed_data, columns=wavelengths.astype(str))
# R_rs_sim.to_csv(outpath + "bio-optics_Rrs_" + AlgaeGroupType + "_simulation_" + iopFnames[0], header=True, sep='\t', index=False)

292
chunks N 1


2026-02-18 16:16:30,597	INFO worker.py:1664 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


(simulate_chunk pid=15760) Parameters([('C_0', <Parameter 'C_0', value=2.180665682352941, bounds=[-inf:inf]>), ('C_1', <Parameter 'C_1', value=0.0, bounds=[-inf:inf]>), ('C_2', <Parameter 'C_2', value=0.0, bounds=[-inf:inf]>), ('C_3', <Parameter 'C_3', value=0.0, bounds=[-inf:inf]>), ('C_4', <Parameter 'C_4', value=0.0, bounds=[-inf:inf]>), ('C_5', <Parameter 'C_5', value=0.0, bounds=[-inf:inf]>), ('C_6', <Parameter 'C_6', value=0.0, bounds=[-inf:inf]>), ('C_7', <Parameter 'C_7', value=0.0, bounds=[-inf:inf]>), ('C_Y', <Parameter 'C_Y', value=0.1880679370446244, bounds=[-inf:inf]>), ('C_ism', <Parameter 'C_ism', value=5.156145813653618, bounds=[-inf:inf]>), ('L_fl_lambda0', <Parameter 'L_fl_lambda0', value=0.0416850426853174, bounds=[-inf:inf]>), ('L_fl_phycocyanin', <Parameter 'L_fl_phycocyanin', value=0.199933967071123, bounds=[-inf:inf]>), ('L_fl_phycoerythrin', <Parameter 'L_fl_phycoerythrin', value=0.0, bounds=[-inf:inf]>), ('b_ratio_C_0', <Parameter 'b_ratio_C_0', value=0.0058, b

RayTaskError(TypeError): [36mray::simulate_chunk()[39m (pid=15760, ip=127.0.0.1)
  File "python\ray\_raylet.pyx", line 1675, in ray._raylet.execute_task
  File "C:\Users\Dagmar\AppData\Local\Temp\ipykernel_10868\1889065741.py", line 34, in simulate_chunk
  File "F:\Anaconda_envs\py310_keras3\lib\site-packages\bio_optics\models\hereon.py", line 225, in forward
    a_res = absorption.a_total(wavelengths=wavelengths,
  File "F:\Anaconda_envs\py310_keras3\lib\site-packages\bio_optics\water\absorption.py", line 880, in a_total
    a_wc = correct_a_phy(a_phy_res=a_phy_res, wavelengths=wavelengths, C_phy=C_phy, A=A, E0=E0, E1=E1, lambda_0_phy=lambda_0_phy, interpolate=interpolate) + \
  File "F:\Anaconda_envs\py310_keras3\lib\site-packages\bio_optics\water\absorption.py", line 815, in correct_a_phy
    if interpolate:
TypeError: __bool__ should return bool, returned numpy.bool_